# Step1：Data Prepare

In [ ]:
# 导入基础库
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.decomposition import PCA
from sklearn.feature_selection import mutual_info_regression
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# 设置图表样式
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

In [ ]:
# 加载训练集和测试集
train_data = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/train.csv')
test_data = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/test.csv')

print(f"训练集形状: {train_data.shape}")
print(f"测试集形状: {test_data.shape}")
print("\n前5行数据:")
train_data.head()

In [ ]:
# 查看目标变量分布
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(train_data['SalePrice'], bins=50, edgecolor='black')
axes[0].set_xlabel('Sale Price')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Sale Price Distribution')

axes[1].hist(np.log1p(train_data['SalePrice']), bins=50, edgecolor='black')
axes[1].set_xlabel('Log(Sale Price)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Log-transformed Sale Price Distribution')

plt.tight_layout()
plt.show()

print(f"SalePrice统计信息:\n{train_data['SalePrice'].describe()}")

# Data Preprocess

In [ ]:
# 保存测试集ID和训练集目标变量
test_ids = test_data['Id']
y_train = train_data['SalePrice'].values
train_data = train_data.drop(['SalePrice'], axis=1)

# 合并数据集
all_data = pd.concat([train_data, test_data], axis=0, ignore_index=True)
print(f"合并后数据集形状: {all_data.shape}")

In [ ]:
# 查看缺失值情况
missing = all_data.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print("缺失值最多的前10个特征:")
print(missing.head(10))

# 对于分类特征，用'None'填充
categorical_features = all_data.select_dtypes(include=['object']).columns
for col in categorical_features:
    all_data[col].fillna('None', inplace=True)

# 对于数值特征，用中位数填充
numerical_features = all_data.select_dtypes(include=['int64', 'float64']).columns
for col in numerical_features:
    if all_data[col].isnull().sum() > 0:
        all_data[col].fillna(all_data[col].median(), inplace=True)

print(f"\n处理后的缺失值总数: {all_data.isnull().sum().sum()}")

In [ ]:
# 创建一些有用的新特征
all_data['TotalSF'] = all_data['TotalBsmtSF'] + all_data['1stFlrSF'] + all_data['2ndFlrSF']
all_data['TotalBath'] = all_data['FullBath'] + 0.5 * all_data['HalfBath'] + \
                         all_data['BsmtFullBath'] + 0.5 * all_data['BsmtHalfBath']
all_data['TotalPorchSF'] = all_data['OpenPorchSF'] + all_data['EnclosedPorch'] + \
                            all_data['3SsnPorch'] + all_data['ScreenPorch']

# 房龄特征
all_data['HouseAge'] = all_data['YrSold'] - all_data['YearBuilt']
all_data['RemodAge'] = all_data['YrSold'] - all_data['YearRemodAdd']

print("新增特征:", ['TotalSF', 'TotalBath', 'TotalPorchSF', 'HouseAge', 'RemodAge'])

In [ ]:
# 使用one-hot编码处理分类变量
all_data = pd.get_dummies(all_data, drop_first=True)
print(f"编码后特征数量: {all_data.shape[1]}")

In [ ]:
# 分离回训练集和测试集
X_train = all_data[:len(train_data)]
X_test = all_data[len(train_data):]

# 对目标变量进行log变换（更符合正态分布）
y_train_log = np.log1p(y_train)

# 划分训练集和验证集
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train_log, 
                                              test_size=0.2, random_state=42)

print(f"训练集: {X_tr.shape}, 验证集: {X_val.shape}")
print(f"测试集: {X_test.shape}")

In [ ]:
# 使用RobustScaler（对异常值更鲁棒）
scaler = RobustScaler()
X_tr_scaled = scaler.fit_transform(X_tr)
X_val_scaled = scaler.transform(X_val)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("特征标准化完成")

# (ADD) Feature Analysis&Choice

## PCA

In [ ]:
# 计算每个特征与目标变量的互信息值
print("计算互信息值（Mutual Information）...")
mi_scores = mutual_info_regression(X_tr_scaled, y_tr, random_state=42)
mi_scores = pd.Series(mi_scores, index=X_train.columns).sort_values(ascending=False)

# 可视化前30个最重要的特征
plt.figure(figsize=(10, 12))
mi_scores.head(30).sort_values().plot(kind='barh', color='#FF6B6B')
plt.xlabel('Mutual Information Score')
plt.title('Top 30 Features by Mutual Information')
plt.tight_layout()
plt.show()

print(f"\n互信息得分统计:")
print(mi_scores.describe())
print(f"\nMI > 0.1 的特征数量: {(mi_scores > 0.1).sum()}")

In [ ]:
# 选择MI值较高的特征（阈值可调整）
mi_threshold = 0.05  # 可以根据实际情况调整
selected_features_mi = mi_scores[mi_scores > mi_threshold].index.tolist()

print(f"选择的特征数量: {len(selected_features_mi)} / {len(mi_scores)}")
print(f"\nMI值最高的前10个特征:")
for i, (feat, score) in enumerate(mi_scores.head(10).items(), 1):
    print(f"{i:2d}. {feat:30s}: {score:.4f}")

In [ ]:
# 对标准化后的数据进行PCA分析
print("执行PCA分析...")
pca = PCA()
pca.fit(X_tr_scaled)

# 计算累积方差解释率
cumsum_variance = np.cumsum(pca.explained_variance_ratio_)

# 可视化方差解释率
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左图：前50个主成分的方差解释率
axes[0].bar(range(1, 51), pca.explained_variance_ratio_[:50], color='#4ECDC4')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Variance Explained Ratio')
axes[0].set_title('Variance Explained by Each PC (Top 50)')
axes[0].grid(alpha=0.3)

# 右图：累积方差解释率
axes[1].plot(range(1, len(cumsum_variance)+1), cumsum_variance, 
             linewidth=2, color='#45B7D1')
axes[1].axhline(y=0.95, color='r', linestyle='--', label='95% Variance')
axes[1].axhline(y=0.99, color='orange', linestyle='--', label='99% Variance')
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Variance Explained')
axes[1].set_title('Cumulative Variance Explained')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# 找出达到95%和99%方差所需的主成分数
n_components_95 = np.argmax(cumsum_variance >= 0.95) + 1
n_components_99 = np.argmax(cumsum_variance >= 0.99) + 1

print(f"\n达到95%方差解释率需要: {n_components_95} 个主成分")
print(f"达到99%方差解释率需要: {n_components_99} 个主成分")
print(f"原始特征数量: {X_tr_scaled.shape[1]}")

In [ ]:
# 使用较少的主成分创建复合特征
n_pca_components = n_components_95  # 使用95%方差的主成分数
pca_final = PCA(n_components=n_pca_components, random_state=42)

# 对训练集、验证集、测试集进行PCA转换
X_tr_pca = pca_final.fit_transform(X_tr_scaled)
X_val_pca = pca_final.transform(X_val_scaled)
X_train_pca = pca_final.transform(X_train_scaled)
X_test_pca = pca_final.transform(X_test_scaled)

print(f"PCA降维后的特征数量: {X_tr_pca.shape[1]}")
print(f"降维比例: {X_tr_pca.shape[1] / X_tr_scaled.shape[1]:.2%}")

In [ ]:
# 分析前5个主成分由哪些原始特征构成
n_top_features = 10
feature_names = X_train.columns

fig, axes = plt.subplots(2, 1, figsize=(18, 10))
axes = axes.flatten()

for i in range(min(6, n_pca_components)):
    # 获取第i个主成分的载荷
    loadings = pca_final.components_[i]
    top_indices = np.argsort(np.abs(loadings))[-n_top_features:][::-1]
    top_features = feature_names[top_indices]
    top_loadings = loadings[top_indices]
    
    # 绘制条形图
    colors = ['#FF6B6B' if x < 0 else '#4ECDC4' for x in top_loadings]
    axes[i].barh(range(n_top_features), top_loadings, color=colors)
    axes[i].set_yticks(range(n_top_features))
    axes[i].set_yticklabels(top_features, fontsize=8)
    axes[i].set_xlabel('Loading')
    axes[i].set_title(f'PC{i+1} (Var: {pca_final.explained_variance_ratio_[i]:.2%})')
    axes[i].axvline(x=0, color='black', linestyle='-', linewidth=0.5)
    axes[i].invert_yaxis()

plt.tight_layout()
plt.show()

print("主成分解释:")
for i in range(min(5, n_pca_components)):
    print(f"\nPC{i+1} (方差贡献: {pca_final.explained_variance_ratio_[i]:.2%}):")
    loadings = pca_final.components_[i]
    top_indices = np.argsort(np.abs(loadings))[-5:][::-1]
    for idx in top_indices:
        print(f"  {feature_names[idx]:30s}: {loadings[idx]:+.3f}")

## MI

In [ ]:
# 计算每个特征与目标变量的互信息值
print("计算互信息值（Mutual Information）...")
print("这可能需要几分钟时间...\n")

mi_scores = mutual_info_regression(X_tr_scaled, y_tr, random_state=42, n_neighbors=5)
mi_scores = pd.Series(mi_scores, index=X_train.columns).sort_values(ascending=False)

# 可视化前30个最重要的特征
plt.figure(figsize=(10, 12))
mi_scores.head(30).sort_values().plot(kind='barh', color='#FF6B6B')
plt.xlabel('Mutual Information Score')
plt.title('Top 30 Features by Mutual Information', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"互信息得分统计:")
print(mi_scores.describe())
print(f"\nMI > 0.1 的特征数量: {(mi_scores > 0.1).sum()}")
print(f"MI > 0.05 的特征数量: {(mi_scores > 0.05).sum()}")
print(f"MI > 0.01 的特征数量: {(mi_scores > 0.01).sum()}")

In [ ]:
# 绘制MI值分布图
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左图：所有特征的MI值分布
axes[0].hist(mi_scores, bins=50, edgecolor='black', alpha=0.7, color='#4ECDC4')
axes[0].axvline(x=0.05, color='r', linestyle='--', linewidth=2, label='Threshold=0.05')
axes[0].set_xlabel('Mutual Information Score')
axes[0].set_ylabel('Number of Features')
axes[0].set_title('Distribution of MI Scores')
axes[0].legend()
axes[0].grid(alpha=0.3)

# 右图：累积特征数量
sorted_mi = np.sort(mi_scores)[::-1]
cumsum_features = np.arange(1, len(sorted_mi) + 1)
axes[1].plot(cumsum_features, sorted_mi, linewidth=2, color='#45B7D1')
axes[1].axhline(y=0.05, color='r', linestyle='--', linewidth=2, label='Threshold=0.05')
axes[1].set_xlabel('Number of Features')
axes[1].set_ylabel('Mutual Information Score')
axes[1].set_title('Cumulative MI Scores')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 测试不同阈值下的模型性能
thresholds = [0.01, 0.03, 0.05, 0.08, 0.1]
threshold_results = {}

print("测试不同MI阈值的效果（使用Ridge快速评估）...\n")

for threshold in thresholds:
    # 选择特征
    selected_features = mi_scores[mi_scores > threshold].index.tolist()
    n_features = len(selected_features)
    
    # 获取对应的特征数据
    X_tr_selected = X_tr_scaled[:, [X_train.columns.get_loc(f) for f in selected_features]]
    X_val_selected = X_val_scaled[:, [X_train.columns.get_loc(f) for f in selected_features]]
    
    # 训练简单的Ridge模型
    model = Ridge(alpha=10, random_state=42)
    model.fit(X_tr_selected, y_tr)
    y_pred = model.predict(X_val_selected)
    
    # 计算性能
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    r2 = r2_score(y_val, y_pred)
    
    threshold_results[threshold] = {
        'n_features': n_features,
        'rmse': rmse,
        'r2': r2
    }
    
    print(f"阈值 {threshold:.2f}: {n_features:3d} 特征, RMSE={rmse:.4f}, R²={r2:.4f}")

# 可视化阈值效果
threshold_df = pd.DataFrame(threshold_results).T

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 特征数量
axes[0].plot(threshold_df.index, threshold_df['n_features'], 
             marker='o', linewidth=2, markersize=8, color='#FF6B6B')
axes[0].set_xlabel('MI Threshold')
axes[0].set_ylabel('Number of Features')
axes[0].set_title('Features vs Threshold')
axes[0].grid(alpha=0.3)

# RMSE
axes[1].plot(threshold_df.index, threshold_df['rmse'], 
             marker='o', linewidth=2, markersize=8, color='#4ECDC4')
axes[1].set_xlabel('MI Threshold')
axes[1].set_ylabel('RMSE')
axes[1].set_title('RMSE vs Threshold')
axes[1].grid(alpha=0.3)

# R²
axes[2].plot(threshold_df.index, threshold_df['r2'], 
             marker='o', linewidth=2, markersize=8, color='#45B7D1')
axes[2].set_xlabel('MI Threshold')
axes[2].set_ylabel('R² Score')
axes[2].set_title('R² vs Threshold')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# 选择最佳阈值
best_threshold = threshold_df['rmse'].idxmin()
print(f"\n推荐阈值: {best_threshold}")
print(f"对应特征数: {threshold_df.loc[best_threshold, 'n_features']:.0f}")

In [ ]:
# 使用最佳阈值选择特征
mi_threshold = best_threshold
selected_features_mi = mi_scores[mi_scores > mi_threshold].index.tolist()

print(f"最终选择的MI阈值: {mi_threshold}")
print(f"选择的特征数量: {len(selected_features_mi)} / {len(mi_scores)}")
print(f"特征保留比例: {len(selected_features_mi)/len(mi_scores):.1%}")

print(f"\nMI值最高的前15个特征:")
for i, (feat, score) in enumerate(mi_scores.head(15).items(), 1):
    print(f"{i:2d}. {feat:40s}: {score:.4f}")

# 准备最终的特征数据
X_tr_final = X_tr_scaled[:, [X_train.columns.get_loc(f) for f in selected_features_mi]]
X_val_final = X_val_scaled[:, [X_train.columns.get_loc(f) for f in selected_features_mi]]
X_train_final = X_train_scaled[:, [X_train.columns.get_loc(f) for f in selected_features_mi]]
X_test_final = X_test_scaled[:, [X_train.columns.get_loc(f) for f in selected_features_mi]]

print(f"\n最终特征维度:")
print(f"训练集: {X_tr_final.shape}")
print(f"验证集: {X_val_final.shape}")
print(f"测试集: {X_test_final.shape}")

# Model Construct

In [ ]:
# 定义三个模型及其详细的超参数网格
models = {
    'Ridge': Ridge(random_state=42),
    'Lasso': Lasso(random_state=42),
    'ElasticNet': ElasticNet(random_state=42)
}

# 详细的超参数网格
param_grids = {
    'Ridge': {
        'alpha': [0.01, 0.1, 0.5, 1, 5, 10, 50, 100, 500, 1000],  # L2正则化强度
        'max_iter': [1000, 5000, 10000],  # 最大迭代次数
        'tol': [1e-4, 1e-3, 1e-2]  # 收敛容忍度
    },
    'Lasso': {
        'alpha': [0.0001, 0.0005, 0.001, 0.005, 0.01, 0.05, 0.1, 0.5],  # L1正则化强度
        'max_iter': [5000, 10000, 20000],  # Lasso通常需要更多迭代
        'tol': [1e-4, 1e-3, 1e-2]
    },
    'ElasticNet': {
        'alpha': [0.0001, 0.0005, 0.001, 0.005, 0.01, 0.05, 0.1],  # 总正则化强度
        'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 0.99],  # L1与L2的比例
        'max_iter': [5000, 10000, 20000],
        'tol': [1e-4, 1e-3]
    }
}

print("超参数说明:")
print("\nalpha - 正则化强度:")
print("  - 值越大，正则化越强，模型越简单（防止过拟合）")
print("  - 值越小，模型越复杂（可能过拟合）")
print("  - Ridge通常需要较大的alpha，Lasso/ElasticNet需要较小的alpha")

print("\nmax_iter - 最大迭代次数:")
print("  - 优化算法的最大迭代步数")
print("  - 如果未收敛，增加此值")

print("\ntol - 收敛容忍度:")
print("  - 判断算法是否收敛的阈值")
print("  - 值越小，收敛条件越严格，精度越高")

print("\nl1_ratio - L1正则化比例（仅ElasticNet）:")
print("  - 0.0 = 纯Ridge (L2)")
print("  - 1.0 = 纯Lasso (L1)")
print("  - 0.5 = L1和L2各占一半")

# Train & Test

In [ ]:
# 对每个模型进行详细的网格搜索
best_models = {}
results = {}

for name, model in models.items():
    print(f"\n{'='*60}")
    print(f"训练 {name} 模型...")
    print(f"搜索空间大小: {np.prod([len(v) for v in param_grids[name].values()])} 种组合")
    
    # 网格搜索
    grid_search = GridSearchCV(
        model, param_grids[name], 
        cv=5, 
        scoring='neg_mean_squared_error',
        n_jobs=-1, 
        verbose=1
    )
    grid_search.fit(X_tr_final, y_tr)
    
    # 保存最佳模型
    best_models[name] = grid_search.best_estimator_
    
    # 预测
    y_val_pred = grid_search.predict(X_val_final)
    
    # 计算指标
    rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
    r2 = r2_score(y_val, y_val_pred)
    mae = mean_absolute_error(y_val, y_val_pred)
    
    results[name] = {
        'best_params': grid_search.best_params_,
        'rmse': rmse,
        'r2': r2,
        'mae': mae,
        'cv_score': np.sqrt(-grid_search.best_score_),
        'train_score': grid_search.score(X_tr_final, y_tr)
    }
    
    print(f"\n最佳参数: {grid_search.best_params_}")
    print(f"验证集 RMSE: {rmse:.4f}")
    print(f"验证集 R²: {r2:.4f}")
    print(f"验证集 MAE: {mae:.4f}")
    print(f"交叉验证 RMSE: {np.sqrt(-grid_search.best_score_):.4f}")
    
    # 检查是否收敛
    if hasattr(grid_search.best_estimator_, 'n_iter_'):
        print(f"实际迭代次数: {grid_search.best_estimator_.n_iter_}")

In [ ]:
# 创建结果对比DataFrame
results_df = pd.DataFrame(results).T
print("\n" + "="*60)
print("模型性能详细对比:")
print("="*60)
print(results_df[['rmse', 'r2', 'mae', 'cv_score']].round(4))

# 可视化超参数对比
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. RMSE对比
axes[0, 0].bar(results_df.index, results_df['rmse'], color=['#FF6B6B', '#4ECDC4', '#45B7D1'])
axes[0, 0].set_ylabel('RMSE')
axes[0, 0].set_title('Validation RMSE Comparison', fontweight='bold')
axes[0, 0].tick_params(axis='x', rotation=45)
axes[0, 0].grid(alpha=0.3, axis='y')

# 2. R²对比
axes[0, 1].bar(results_df.index, results_df['r2'], color=['#FF6B6B', '#4ECDC4', '#45B7D1'])
axes[0, 1].set_ylabel('R² Score')
axes[0, 1].set_title('R² Score Comparison', fontweight='bold')
axes[0, 1].tick_params(axis='x', rotation=45)
axes[0, 1].grid(alpha=0.3, axis='y')

# 3. CV vs Validation RMSE
x = np.arange(len(results_df))
width = 0.35
axes[1, 0].bar(x - width/2, results_df['cv_score'], width, label='CV RMSE', color='#FFB6C1')
axes[1, 0].bar(x + width/2, results_df['rmse'], width, label='Val RMSE', color='#87CEEB')
axes[1, 0].set_ylabel('RMSE')
axes[1, 0].set_title('Cross-Validation vs Validation RMSE', fontweight='bold')
axes[1, 0].set_xticks(x)
axes[1, 0].set_xticklabels(results_df.index, rotation=45)
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3, axis='y')

# 4. 最佳alpha值对比
alphas = [results[name]['best_params']['alpha'] for name in results_df.index]
axes[1, 1].bar(results_df.index, alphas, color=['#FF6B6B', '#4ECDC4', '#45B7D1'])
axes[1, 1].set_ylabel('Alpha (log scale)')
axes[1, 1].set_title('Best Alpha Values', fontweight='bold')
axes[1, 1].set_yscale('log')
axes[1, 1].tick_params(axis='x', rotation=45)
axes[1, 1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# 找出最佳模型
best_model_name = results_df['rmse'].idxmin()
print(f"\n{'='*60}")
print(f"最佳模型: {best_model_name}")
print(f"最佳RMSE: {results_df.loc[best_model_name, 'rmse']:.4f}")
print(f"最佳R²: {results_df.loc[best_model_name, 'r2']:.4f}")
print(f"{'='*60}")

In [ ]:
print("超参数优化总结:\n")

for name in results.keys():
    print(f"\n{name} 模型:")
    print("-" * 40)
    params = results[name]['best_params']
    
    # Alpha分析
    if 'alpha' in params:
        alpha_val = params['alpha']
        print(f"✓ Alpha = {alpha_val}")
        if alpha_val < 0.01:
            print("  → 弱正则化，模型较复杂，注意过拟合")
        elif alpha_val < 1:
            print("  → 中等正则化，平衡复杂度和泛化能力")
        else:
            print("  → 强正则化，模型简单，防止过拟合")
    
    # L1 ratio分析（ElasticNet）
    if 'l1_ratio' in params:
        l1_ratio = params['l1_ratio']
        print(f"✓ L1_ratio = {l1_ratio}")
        if l1_ratio < 0.3:
            print("  → 偏向Ridge (L2)，保留大部分特征")
        elif l1_ratio < 0.7:
            print("  → 平衡L1和L2，适度特征选择")
        else:
            print("  → 偏向Lasso (L1)，强特征选择")
    
    # Max_iter和tol分析
    if 'max_iter' in params:
        print(f"✓ Max_iter = {params['max_iter']}")
    if 'tol' in params:
        print(f"✓ Tolerance = {params['tol']}")
        if params['tol'] <= 1e-4:
            print("  → 严格收敛条件，高精度")
        else:
            print("  → 宽松收敛条件，快速训练")

# Plot Result

In [ ]:
# 绘制预测值vs真实值
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, (name, model) in enumerate(best_models.items()):
    y_pred = model.predict(X_val_final)
    
    axes[idx].scatter(y_val, y_pred, alpha=0.5, s=20, color='#4ECDC4')
    axes[idx].plot([y_val.min(), y_val.max()], 
                   [y_val.min(), y_val.max()], 
                   'r--', lw=2, label='Perfect Prediction')
    axes[idx].set_xlabel('Actual Log(Price)', fontsize=11)
    axes[idx].set_ylabel('Predicted Log(Price)', fontsize=11)
    axes[idx].set_title(f'{name}\nR² = {results[name]["r2"]:.4f}', fontweight='bold')
    axes[idx].legend()
    axes[idx].grid(alpha=0.3)
    
plt.tight_layout()
plt.show()

In [ ]:
# 绘制残差图
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for idx, (name, model) in enumerate(best_models.items()):
    y_pred = model.predict(X_val_final)
    residuals = y_val - y_pred
    
    # 残差散点图
    axes[0, idx].scatter(y_pred, residuals, alpha=0.5, s=20, color='#4ECDC4')
    axes[0, idx].axhline(y=0, color='r', linestyle='--', lw=2)
    axes[0, idx].set_xlabel('Predicted Log(Price)', fontsize=10)
    axes[0, idx].set_ylabel('Residuals', fontsize=10)
    axes[0, idx].set_title(f'{name} - Residual Plot', fontweight='bold')
    axes[0, idx].grid(alpha=0.3)
    
    # 残差分布直方图
    axes[1, idx].hist(residuals, bins=30, edgecolor='black', alpha=0.7, color='#FF6B6B')
    axes[1, idx].axvline(x=0, color='r', linestyle='--', lw=2)
    axes[1, idx].set_xlabel('Residuals', fontsize=10)
    axes[1, idx].set_ylabel('Frequency', fontsize=10)
    axes[1, idx].set_title(f'{name} - Residual Distribution', fontweight='bold')
    axes[1, idx].grid(alpha=0.3, axis='y')
    
plt.tight_layout()
plt.show()

In [ ]:
# 分析三个模型的特征系数
fig, axes = plt.subplots(1, 3, figsize=(15, 10))

for idx, (name, model) in enumerate(best_models.items()):
    coef = pd.Series(model.coef_, index=selected_features_mi)
    
    if name == 'Lasso':
        # Lasso: 只显示非零系数
        non_zero_coef = coef[coef != 0].sort_values(key=abs, ascending=False)
        top_coef = non_zero_coef.head(25)
        
        axes[idx].barh(range(len(top_coef)), top_coef.values, color='#4ECDC4')
        axes[idx].set_yticks(range(len(top_coef)))
        axes[idx].set_yticklabels(top_coef.index, fontsize=8)
        axes[idx].set_xlabel('Coefficient Value', fontsize=10)
        axes[idx].set_title(f'{name}\n{len(non_zero_coef)} features selected', fontweight='bold')
        axes[idx].axvline(x=0, color='black', linestyle='-', linewidth=0.5)
        axes[idx].invert_yaxis()
        axes[idx].grid(alpha=0.3, axis='x')
    else:
        # Ridge和ElasticNet: 显示绝对值最大的25个系数
        top_coef = coef.abs().sort_values(ascending=False).head(25)
        top_coef = coef[top_coef.index]
        
        colors = ['#FF6B6B' if x < 0 else '#4ECDC4' for x in top_coef.values]
        axes[idx].barh(range(len(top_coef)), top_coef.values, color=colors)
        axes[idx].set_yticks(range(len(top_coef)))
        axes[idx].set_yticklabels(top_coef.index, fontsize=8)
        axes[idx].set_xlabel('Coefficient Value', fontsize=10)
        axes[idx].set_title(f'{name}\nTop 25 features by |coef|', fontweight='bold')
        axes[idx].axvline(x=0, color='black', linestyle='-', linewidth=0.5)
        axes[idx].invert_yaxis()
        axes[idx].grid(alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

# 打印Lasso特征选择结果
lasso_model = best_models['Lasso']
lasso_coef = pd.Series(lasso_model.coef_, index=selected_features_mi)
lasso_non_zero = lasso_coef[lasso_coef != 0]

print(f"\nLasso特征选择结果:")
print(f"原始特征数: {len(selected_features_mi)}")
print(f"Lasso选择: {len(lasso_non_zero)} 个特征")
print(f"特征稀疏度: {(1 - len(lasso_non_zero)/len(selected_features_mi)):.1%}")

In [ ]:
# 绘制不同alpha值下的学习曲线
from sklearn.model_selection import learning_curve

def plot_learning_curve(estimator, X, y, title):
    train_sizes, train_scores, val_scores = learning_curve(
        estimator, X, y, cv=5, n_jobs=-1,
        train_sizes=np.linspace(0.1, 1.0, 10),
        scoring='neg_mean_squared_error'
    )
    
    train_rmse = np.sqrt(-train_scores)
    val_rmse = np.sqrt(-val_scores)
    
    train_rmse_mean = np.mean(train_rmse, axis=1)
    train_rmse_std = np.std(train_rmse, axis=1)
    val_rmse_mean = np.mean(val_rmse, axis=1)
    val_rmse_std = np.std(val_rmse, axis=1)
    
    plt.plot(train_sizes, train_rmse_mean, label='Training RMSE', 
             color='#FF6B6B', linewidth=2)
    plt.fill_between(train_sizes, train_rmse_mean - train_rmse_std,
                     train_rmse_mean + train_rmse_std, alpha=0.2, color='#FF6B6B')
    
    plt.plot(train_sizes, val_rmse_mean, label='Validation RMSE', 
             color='#4ECDC4', linewidth=2)
    plt.fill_between(train_sizes, val_rmse_mean - val_rmse_std,
                     val_rmse_mean + val_rmse_std, alpha=0.2, color='#4ECDC4')
    
    plt.xlabel('Training Set Size', fontsize=11)
    plt.ylabel('RMSE', fontsize=11)
    plt.title(title, fontweight='bold')
    plt.legend()
    plt.grid(alpha=0.3)

# 为最佳模型绘制学习曲线
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, (name, model) in enumerate(best_models.items()):
    plt.sca(axes[idx])
    plot_learning_curve(model, X_tr_final, y_tr, f'{name} Learning Curve')

plt.tight_layout()
plt.show()

# Optimize & Review

In [ ]:
# 使用全部训练数据训练最佳模型
final_model = best_models[best_model_name]
final_model.fit(X_train_final, y_train_log)

print(f"最终模型: {best_model_name}")
print(f"使用特征数: {X_train_final.shape[1]}")
print(f"MI阈值: {mi_threshold}")
print(f"最佳参数: {results[best_model_name]['best_params']}")

In [ ]:
# 对测试集进行预测
y_test_pred_log = final_model.predict(X_test_final)
y_test_pred = np.expm1(y_test_pred_log)  # 转换回原始尺度

# 创建提交文件
submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': y_test_pred
})

submission.to_csv('submission.csv', index=False)
print("预测完成，已保存至 submission.csv")
print(f"\n预测的房价统计:")
print(submission['SalePrice'].describe())

# 可视化预测分布
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左图：预测价格分布
axes[0].hist(y_test_pred, bins=50, edgecolor='black', alpha=0.7, color='#4ECDC4')
axes[0].axvline(x=y_test_pred.mean(), color='r', linestyle='--', 
                linewidth=2, label=f'Mean: ${y_test_pred.mean():,.0f}')
axes[0].set_xlabel('Predicted Sale Price', fontsize=11)
axes[0].set_ylabel('Frequency', fontsize=11)
axes[0].set_title('Distribution of Predicted Sale Prices', fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3, axis='y')

# 右图：训练集和预测对比
axes[1].hist(y_train, bins=50, alpha=0.5, label='Training Data', color='#FF6B6B')
axes[1].hist(y_test_pred, bins=50, alpha=0.5, label='Test Predictions', color='#4ECDC4')
axes[1].set_xlabel('Sale Price', fontsize=11)
axes[1].set_ylabel('Frequency', fontsize=11)
axes[1].set_title('Training vs Predicted Distribution', fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
print("="*70)
print("房价预测模型 - 综合分析报告")
print("="*70)

print("\n【1. 数据集概况】")
print(f"  • 训练样本数: {len(train_data)}")
print(f"  • 原始特征数: {train_data.shape[1] - 1}")
print(f"  • 编码后特征数: {X_train.shape[1]}")
print(f"  • 目标变量范围: ${y_train.min():,.0f} - ${y_train.max():,.0f}")

print("\n【2. 特征工程效果】")
print(f"  • 互信息分析:")
print(f"    ├─ 最佳MI阈值: {mi_threshold}")
print(f"    ├─ 选择特征数: {len(selected_features_mi)}")
print(f"    ├─ 特征保留率: {len(selected_features_mi)/X_train.shape[1]:.1%}")
print(f"    └─ 降维效果: {X_train.shape[1]} → {len(selected_features_mi)}")

print("\n【3. 模型性能对比】")
for name in results.keys():
    print(f"\n  {name}:")
    print(f"    ├─ 验证集RMSE: {results[name]['rmse']:.4f}")
    print(f"    ├─ 验证集R²: {results[name]['r2']:.4f}")
    print(f"    ├─ 验证集MAE: {results[name]['mae']:.4f}")
    print(f"    ├─ 交叉验证RMSE: {results[name]['cv_score']:.4f}")
    print(f"    └─ 最佳Alpha: {results[name]['best_params']['alpha']}")

print(f"\n【4. 最佳模型】")
print(f"  • 模型类型: {best_model_name}")
print(f"  • 验证集RMSE: {results[best_model_name]['rmse']:.4f}")
print(f"  • 验证集R²: {results[best_model_name]['r2']:.4f}")
print(f"  • 验证集MAE: {results[best_model_name]['mae']:.4f}")
print(f"  • 优化参数:")
for param, value in results[best_model_name]['best_params'].items():
    print(f"      {param}: {value}")

print("\n【5. 超参数优化见解】")
print("  Alpha (正则化强度):")
alphas_comparison = {name: results[name]['best_params']['alpha'] for name in results.keys()}
print(f"    • Ridge: {alphas_comparison['Ridge']:.4f} (较大，L2正则化需要强惩罚)")
print(f"    • Lasso: {alphas_comparison['Lasso']:.4f} (较小，L1易产生稀疏性)")
print(f"    • ElasticNet: {alphas_comparison['ElasticNet']:.4f} (介于两者之间)")

if 'l1_ratio' in results['ElasticNet']['best_params']:
    l1_ratio = results['ElasticNet']['best_params']['l1_ratio']
    print(f"\n  L1_ratio (ElasticNet):")
    print(f"    • 最佳值: {l1_ratio}")
    print(f"    • L1占比: {l1_ratio:.1%}, L2占比: {1-l1_ratio:.1%}")

print("\n【6. 特征选择效果】")
# 分析Lasso的特征选择
lasso_coef = pd.Series(best_models['Lasso'].coef_, index=selected_features_mi)
lasso_non_zero = lasso_coef[lasso_coef != 0]
print(f"  • Lasso特征选择:")
print(f"    ├─ MI筛选后: {len(selected_features_mi)} 个特征")
print(f"    ├─ Lasso再选: {len(lasso_non_zero)} 个特征")
print(f"    └─ 总体稀疏度: {(1 - len(lasso_non_zero)/X_train.shape[1]):.1%}")

print("\n【7. MI特征选择优势】")
print("  • ✓ 移除了低相关性特征，降低噪声")
print("  • ✓ 减少了特征维度，加快训练速度")
print("  • ✓ 保留了与目标变量强相关的特征")
print("  • ✓ 避免了PCA带来的解释性损失")
print(f"  • ✓ 特征数从{X_train.shape[1]}降到{len(selected_features_mi)}，降幅{(1-len(selected_features_mi)/X_train.shape[1]):.1%}")

print("\n【8. 改进建议】")
print("  • 尝试多项式特征交互")
print("  • 考虑集成学习方法（Stacking多个模型）")
print("  • 处理潜在的异常值")
print("  • 尝试非线性变换（如Box-Cox）")
print("  • 可以尝试不同的MI阈值进一步优化")

print("\n" + "="*70)
print(f"预测完成！提交文件已保存为 submission.csv")
print("="*70)